# Drug Discovery GRPO — Kaggle Runbook

End-to-end on a Kaggle GPU notebook (T4 x1 / x2 or P100):

1. Clone the repo from GitHub
2. Install all dependencies (TRL, Unsloth 4-bit, RDKit, FastAPI, etc.)
3. Build the disease/target dataset (`prepare_dataset.py`)
4. Boot the FastAPI env server inside the notebook
5. Run multi-disease GRPO training (`train.py`)
6. Evaluate on the held-out test split (`evaluate.py`)
7. Run inference on a brand-new (unseen) disease (`infer.py`)

> Before running: in **Kaggle Notebook → Settings**, enable **Internet** and pick an
> **Accelerator** (`GPU T4 x2` recommended). Then run the cells in order.

## 0. Choose run knobs

These are the only values you should typically change. They are exported as env
vars and consumed by the CLI flags below. Defaults are tuned for a Kaggle T4.

In [36]:
import os

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/VasuBB/drug-discovery-sim-env.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'akshat_optimal')
WORK_DIR = '/kaggle/working/drug-discovery-sim-env' if os.path.isdir('/kaggle') else os.path.abspath('drug-discovery-sim-env')

# Dataset prep
NUM_DISEASES = 100    # bump to 5000+ for the full dataset
TEST_FRACTION = float(os.environ.get('TEST_FRACTION', '0.10'))
MIN_DRUGGABILITY = float(os.environ.get('MIN_DRUGGABILITY', '0.30'))
KNOWN_DRUGS_PER_TARGET = int(os.environ.get('KNOWN_DRUGS_PER_TARGET', '8'))

# Training
MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen/Qwen2.5-0.5B-Instruct')
NUM_TRAIN_STEPS = int(os.environ.get('NUM_TRAIN_STEPS', '10'))
GROUP_SIZE = int(os.environ.get('GROUP_SIZE', '4'))
ENV_PORT = int(os.environ.get('ENV_PORT', '8000'))
BASE_URL = f'http://127.0.0.1:{ENV_PORT}'

# Evaluation / inference
EVAL_LIMIT = int(os.environ.get('EVAL_LIMIT', '20'))          # cap test diseases (None = use all)
INFER_DISEASE = os.environ.get('INFER_DISEASE', 'Idiopathic pulmonary fibrosis')

print('Work dir:', WORK_DIR)
print('Model:', MODEL_NAME, '| GRPO steps:', NUM_TRAIN_STEPS, '| group size:', GROUP_SIZE)

Work dir: /kaggle/working/drug-discovery-sim-env
Model: Qwen/Qwen2.5-0.5B-Instruct | GRPO steps: 10 | group size: 4


## 1. Clone the repo

In [38]:
import os, subprocess, sys

def _run(*args):
    subprocess.check_call(list(args))

is_repo = os.path.isdir(os.path.join(WORK_DIR, ".git"))
# print(is_repo)

if not is_repo:
    if os.path.isdir(WORK_DIR):
        import shutil; shutil.rmtree(WORK_DIR)
    _run("git", "clone", "--depth", "1", "--single-branch",
         "--branch", REPO_BRANCH, REPO_URL, WORK_DIR)
else:
    _run("git", "-C", WORK_DIR, "remote", "set-url", "origin", REPO_URL)
    _run("git", "-C", WORK_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH)
    _run("git", "-C", WORK_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}")
    _run("git", "-C", WORK_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}")
    # _run("git", "-C", WORK_DIR, "clean", "-fdx")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

head = subprocess.check_output(
    ["git", "-C", WORK_DIR, "log", "-1", "--oneline"]
).decode().strip()
print("Working in", os.getcwd())
print("On branch:", REPO_BRANCH, "@", head)

Branch 'akshat_optimal' set up to track remote branch 'akshat_optimal' from 'origin'.
Your branch is up to date with 'origin/akshat_optimal'.
HEAD is now at 226a998 a
Working in /kaggle/working/drug-discovery-sim-env
On branch: akshat_optimal @ 226a998 a


From https://github.com/VasuBB/drug-discovery-sim-env
 * branch            akshat_optimal -> FETCH_HEAD
Reset branch 'akshat_optimal'


In [42]:
import shutil

SRC  = "/kaggle/working/outputs1"
DEST = "/kaggle/working/drug-discovery-sim-env/outputs"

shutil.copytree(SRC, DEST, dirs_exist_ok=True)
print("Copied", SRC, "->", DEST)

Copied /kaggle/working/outputs1 -> /kaggle/working/drug-discovery-sim-env/outputs


In [37]:
import os, subprocess, sys

if not os.path.isdir(WORK_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, WORK_DIR])
else:
    subprocess.check_call(['git', '-C', WORK_DIR, 'fetch', '--depth', '1', 'origin', REPO_BRANCH])
    subprocess.check_call(['git', '-C', WORK_DIR, 'checkout', REPO_BRANCH])
    subprocess.check_call(['git', '-C', WORK_DIR, 'reset', '--hard', f'origin/{REPO_BRANCH}'])

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('Working in', os.getcwd())

Your branch and 'origin/akshat_optimal' have diverged,
and have 1 and 1 different commits each, respectively.
  (use "git pull" to merge the remote branch into yours)
HEAD is now at 226a998 a
Working in /kaggle/working/drug-discovery-sim-env


From https://github.com/VasuBB/drug-discovery-sim-env
 * branch            akshat_optimal -> FETCH_HEAD
 + 4bfec88...226a998 akshat_optimal -> origin/akshat_optimal  (forced update)
Already on 'akshat_optimal'


## 2. Install dependencies

Unsloth ships its own torch/triton wheels, so we install it first and let pip
resolve everything else around it. RDKit and TRL come from the `[chem]` and
`[training]` extras.

In [ ]:
%%capture
# Modern TRL (>=0.18) wires in mergekit at import time and uses the openenv
# rollout API our trainer relies on, so we install mergekit + the [training]
# extra. Unsloth is optional; our trainer falls back to plain HF if the
# installed torch wheel isn't compatible with Unsloth's cpp extensions.
!pip install --quiet --upgrade pip
!pip install --quiet 'trl>=0.18' 'transformers>=4.44' 'accelerate>=0.33' \
    'datasets>=2.20' 'peft>=0.11' 'bitsandbytes>=0.43' 'mergekit>=0.0.5'
!pip install --quiet 'rdkit>=2024.3.1' rank-bm25 'sentence-transformers>=2.7'
!pip install --quiet 'fastapi>=0.115' 'uvicorn[standard]>=0.30' \
    'pydantic>=2.7' 'pydantic-settings>=2.2' 'PyYAML>=6.0' requests numpy networkx
!pip install --quiet 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' || echo '[warn] Unsloth not installed; falling back to plain HF backend'
!pip install --quiet -e .[training,test,chem]

In [ ]:
import importlib, torch
for m in ['transformers', 'trl', 'mergekit', 'datasets', 'peft', 'accelerate', 'rdkit', 'fastapi', 'uvicorn', 'drug_discovery_env']:
    try:
        mod = importlib.import_module(m)
        print(f'{m}: OK ({getattr(mod, "__version__", "")})')
    except Exception as e:
        print(f'{m}: FAIL -', e)
print('CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
import importlib, torch
for m in ['transformers', 'trl', 'datasets', 'peft', 'accelerate', 'rdkit', 'fastapi', 'uvicorn', 'drug_discovery_env']:
    try:
        mod = importlib.import_module(m)
        print(f'{m}: OK ({getattr(mod, "__version__", "")})')
    except Exception as e:
        print(f'{m}: FAIL -', e)
print('CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Build the disease / target / known-drugs dataset

One-time fetch from Open Targets + ChEMBL. Set `NUM_DISEASES` higher (e.g. 5500)
for a full run. Internet must be enabled.

In [39]:
!python -m drug_discovery_env.scripts.prepare_dataset \
    --num-diseases 10 \
    --test-fraction {TEST_FRACTION} \
    --min-druggability {MIN_DRUGGABILITY} \
    --known-drugs-per-target {KNOWN_DRUGS_PER_TARGET}


[prepare_dataset] fetching up to 10 diseases from Open Targets...
[prepare_dataset] wrote /kaggle/working/drug-discovery-sim-env/data/diseases.jsonl (9 train + 1 test) and manifest /kaggle/working/drug-discovery-sim-env/data/diseases.manifest.json


In [10]:

import json, pathlib
manifest = json.loads(pathlib.Path('data/diseases.manifest.json').read_text())
manifest

{'rows': 100,
 'n_train': 90,
 'n_test': 10,
 'test_fraction': 0.1,
 'seed': 42,
 'min_druggability': 0.3,
 'chembl_known_drugs_per_target': 8,
 'fetched_at': '2026-04-26T07:54:45.207044+00:00',
 'sha256': '94370e5a47c025a2774e9a6561c4c92a3673243fc0bfb17297e65ed64b7bd36c',
 'cache_path': '/kaggle/working/drug-discovery-sim-env/data/diseases.jsonl'}

## 4. Boot the FastAPI env server (background)

We launch uvicorn in a subprocess and wait for `/health` to return 200 before
moving on. Logs are streamed to `outputs/server.log`.

In [ ]:
!pip install -q openenv_core

In [41]:
import os, subprocess, time, requests

os.makedirs('outputs', exist_ok=True)
log_handle = open('outputs/server.log', 'w', buffering=1)
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'drug_discovery_env.server.app:app',
     '--host', '127.0.0.1', '--port', str(ENV_PORT), '--log-level', 'warning'],
    stdout=log_handle, stderr=subprocess.STDOUT,
)

for attempt in range(60):
    try:
        r = requests.get(f'{BASE_URL}/health', timeout=2.0)
        if r.status_code == 200:
            print('env server up:', r.json())
            break
    except Exception:
        pass
    time.sleep(1.0)
else:
    raise RuntimeError('env server failed to start; check outputs/server.log')

env server up: {'status': 'healthy'}


## 5. GRPO training

Multi-disease live-rollout training. Per-turn JSONL traces (tool calls,
literature queries, reasoning, reward breakdown) are written under
`outputs/grpo/logs/<run-id>/`.

In [40]:
# replace cell 9 (prepare_dataset)
DATA_PATH = '/kaggle/working/drug-discovery-sim-env/data/diseases.jsonl'
MANIFEST_PATH = '/kaggle/working/drug-discovery-sim-env/data/diseases.manifest.json'

!python -m drug_discovery_env.scripts.prepare_dataset \
    --num-diseases 10 \
    --test-fraction {TEST_FRACTION} \
    --min-druggability {MIN_DRUGGABILITY} \
    --known-drugs-per-target {KNOWN_DRUGS_PER_TARGET} \
    --output {DATA_PATH} \
    --manifest {MANIFEST_PATH}

[prepare_dataset] fetching up to 10 diseases from Open Targets...
[prepare_dataset] wrote /kaggle/working/drug-discovery-sim-env/data/diseases.jsonl (9 train + 1 test) and manifest /kaggle/working/drug-discovery-sim-env/data/diseases.manifest.json


In [19]:
!python -m drug_discovery_env.scripts.train \
    --base-url {BASE_URL} \
    --model {MODEL_NAME} \
    --num-train-steps {NUM_TRAIN_STEPS} \
    --group-size {GROUP_SIZE} \
    --output-dir outputs/grpo \
    --log-dir outputs/grpo/logs \
    --run-id kaggle-run

[dataset] reading /kaggle/working/drug-discovery-sim-env/data/diseases.jsonl (29829 bytes)
[train] base_url=http://127.0.0.1:8000  model=Qwen/Qwen2.5-0.5B-Instruct
        train_diseases=90  test_diseases=10
        steps=10  group=4
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-04-26 08:11:48.598323: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777191108.622191    1798 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777191108.629957    1798 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777191108.652036    1798 computation_placer.cc:177] computation p

In [20]:
import pathlib
for p in sorted(pathlib.Path('outputs/grpo').glob('*'))[:20]:
    print(p)
print('---')
csv_path = pathlib.Path('outputs/grpo/logs/kaggle-run/runs.csv')
if csv_path.exists():
    print(csv_path.read_text()[:2000])

outputs/grpo/README.md
outputs/grpo/adapter_config.json
outputs/grpo/adapter_model.safetensors
outputs/grpo/added_tokens.json
outputs/grpo/chat_template.jinja
outputs/grpo/checkpoint-10
outputs/grpo/logs
outputs/grpo/merges.txt
outputs/grpo/special_tokens_map.json
outputs/grpo/tokenizer.json
outputs/grpo/tokenizer_config.json
outputs/grpo/training_args.bin
outputs/grpo/vocab.json
---
episode_id,disease,steps,terminal_reward,total_reward,stage_completed,budget_remaining_frac,oversight_violations,terminated_reason
1777189540590-2af55cf1,"Type 2 Diabetes",1,0.0000,0.1581,0,0.9943,0,plan_exhausted (1 actions)
1777189540625-b506c6ca,"Type 2 Diabetes",5,0.0000,0.1290,0,0.9615,0,plan_exhausted (5 actions)
1777189541823-8b423b40,"Type 2 Diabetes",0,0.0000,0.0000,0,1.0000,0,plan_exhausted (0 actions)
1777189541828-6f621112,"Type 2 Diabetes",2,0.0000,0.1528,0,0.9875,0,plan_exhausted (2 actions)
1777189541927-4bf9a00c,"Type 2 Diabetes",4,0.0000,0.1647,0,0.9773,0,plan_exhausted (4 actions)
1777189

## 6. Evaluate on the held-out test split

Computes the full panel: env reward, ADMET pass, oversight violations, budget
remaining, mean reasoning depth, and ChEMBL Tanimoto-to-known-drugs (precision@1
vs. cached known compounds for each test disease's target).

In [43]:
!python -m drug_discovery_env.scripts.evaluate \
    --base-url {BASE_URL} \
    --checkpoint outputs/grpo \
    --limit 5

[evaluate] checkpoint=outputs/grpo  test_diseases=1
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-04-26 09:15:58.581499: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777194958.606582    2501 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777194958.615733    2501 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777194958.637102    2501 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777194958.637157    2501 computation_placer.cc:177] computation placer a

In [44]:
import json, pathlib
report = json.loads(pathlib.Path('outputs/eval/report.json').read_text())
report

{'n_test_diseases': 1,
 'mean_terminal_reward': 0.0,
 'mean_total_reward': 0.14720770740521888,
 'stage_completion_rate': 0.0,
 'admet_pass_rate': 0.0,
 'oversight_violation_rate': 0.0,
 'mean_oversight_violations': 0.0,
 'mean_budget_remaining_frac': 0.9501480000000001,
 'mean_reasoning_depth': 0.10181347307639543,
 'mean_tanimoto_to_known': 0.0,
 'mean_tanimoto_to_known_max': 0.0,
 'precision_at_1': 0.0,
 'mean_steps': 50.0,
 'per_disease_path': '/kaggle/working/drug-discovery-sim-env/outputs/eval/per_disease.jsonl'}

## 7. Inference on an unseen disease

Pass any disease string. If it isn't in the cache the env falls back to live
Open Targets for the target lookup, then drives a full campaign with the
trained checkpoint.

In [ ]:
!python -m drug_discovery_env.scripts.infer \
    --base-url {BASE_URL} \
    --checkpoint outputs/grpo \
    --disease "Type 2 Diabetes" \
    --out-dir outputs/infer

import json, pathlib, re
slug = re.sub(r'[^a-z0-9]+', '-', INFER_DISEASE.lower()).strip('-')
summary = json.loads(pathlib.Path(f'outputs/infer/{slug}.json').read_text())
summary

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-04-26 09:31:58.662447: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777195918.687739    2580 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777195918.696247    2580 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777195918.720113    2580 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777195918.720176    2580 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid li

## 7b. Training plots + results bundle

The GRPO trainer writes **live plots** (loss, mean reward +/- std, per-reward
component) into `outputs/grpo/plots/` after every optimizer step via
`TrainingPlotsCallback`, plus a tidy `outputs/grpo/metrics.csv` with every
logged scalar. The cell below

1. displays the live plots inline (proof of a real run);
2. plots per-episode terminal/total rewards from `outputs/grpo/logs/<run-id>/runs.csv`
   (which now also carries `nominated_compound_id` + `nominated_smiles` for
   each rollout) and copies it under `outputs/plots/`;
3. zips the entire results folder for one-click download.

In [ ]:
import csv, pathlib, shutil
import matplotlib.pyplot as plt
from IPython.display import Image, display

PLOT_DIR = pathlib.Path('outputs/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

LIVE_PLOTS = pathlib.Path('outputs/grpo/plots')
print('[plot] live training plots dir:', LIVE_PLOTS)
if LIVE_PLOTS.exists():
    for png in sorted(LIVE_PLOTS.glob('*.png')):
        shutil.copy2(png, PLOT_DIR / png.name)
        print(f'  [ok]  {png.name} -> {PLOT_DIR / png.name}')
        display(Image(filename=str(png)))
else:
    print('[warn] no live plots found - did the GRPO trainer run with TrainingPlotsCallback?')

metrics_csv = pathlib.Path('outputs/grpo/metrics.csv')
if metrics_csv.exists():
    head = metrics_csv.read_text(encoding='utf-8').splitlines()[:5]
    print('\n[plot] outputs/grpo/metrics.csv head:')
    for line in head:
        print('  ', line)

runs_dir = pathlib.Path('outputs/grpo/logs')
runs_csv = next(iter(sorted(runs_dir.glob('*/runs.csv'))), None)
if runs_csv and runs_csv.exists():
    eps, terminal, total, nominated = [], [], [], 0
    with runs_csv.open('r', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        for i, row in enumerate(reader):
            eps.append(i + 1)
            terminal.append(float(row['terminal_reward']))
            total.append(float(row['total_reward']))
            if (row.get('nominated_compound_id') or '').strip():
                nominated += 1
    print(f'\n[plot] {runs_csv}: {len(eps)} episodes ({nominated} with a nominated compound)')

    def _rolling(xs, k=10):
        if len(xs) < 2:
            return xs
        k = min(k, len(xs))
        return [sum(xs[max(0, i - k + 1):i + 1]) / len(xs[max(0, i - k + 1):i + 1]) for i in range(len(xs))]

    if eps:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(eps, total, alpha=0.35, color='#16a085', label='total reward (raw)')
        ax.plot(eps, _rolling(total, 10), color='#16a085', linewidth=2, label='total reward (rolling-10)')
        ax.plot(eps, terminal, alpha=0.35, color='#e67e22', label='terminal reward (raw)')
        ax.plot(eps, _rolling(terminal, 10), color='#e67e22', linewidth=2, label='terminal reward (rolling-10)')
        ax.set_xlabel('episode'); ax.set_ylabel('reward')
        ax.set_title(f'Per-episode rewards across {len(eps)} training rollouts')
        ax.legend(loc='best'); ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(PLOT_DIR / 'episode_rewards.png', dpi=150)
        plt.show()
else:
    print('[warn] no runs.csv found - no per-episode reward log.')

print('\nFigures in outputs/plots/:', sorted(p.name for p in PLOT_DIR.glob('*.png')))


import os, datetime

ON_KAGGLE = os.path.isdir('/kaggle/working')
DEST_DIR = pathlib.Path('/kaggle/working') if ON_KAGGLE else pathlib.Path.cwd()
DEST_DIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
bundle_name = f'drug_discovery_results_{stamp}'
staging = pathlib.Path('outputs') / '_bundle' / bundle_name
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True, exist_ok=True)

def _copy(src, dst_rel):
    src = pathlib.Path(src)
    if not src.exists():
        print(f'  [skip] {src} (missing)')
        return
    dst = staging / dst_rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    print(f'  [ok]   {src} -> {dst.relative_to(staging)}')

print('\nStaging artefacts...')
_copy('data/diseases.jsonl',            'data/diseases.jsonl')
_copy('data/diseases.manifest.json',    'data/diseases.manifest.json')
_copy('outputs/plots',                  'plots')
_copy('outputs/grpo/plots',             'grpo/plots')
_copy('outputs/grpo/metrics.csv',       'grpo/metrics.csv')
_copy('outputs/eval/report.json',       'eval/report.json')
_copy('outputs/eval/per_disease.jsonl', 'eval/per_disease.jsonl')
_copy('outputs/infer',                  'infer')
_copy('outputs/grpo/logs',              'grpo/logs')
_copy('outputs/grpo/trainer_state.json','grpo/trainer_state.json')
_copy('outputs/grpo/all_results.json',  'grpo/all_results.json')
_copy('outputs/grpo/train_results.json','grpo/train_results.json')
_copy('outputs/server.log',             'server.log')
_copy('outputs/grpo/train.log',         'grpo/train.log')
_copy('outputs/eval/run.log',           'eval/run.log')
_copy('outputs/infer/run.log',          'infer/run.log')

for nb in [pathlib.Path('/kaggle/working/__notebook_source__.ipynb'),
           pathlib.Path('notebooks/kaggle_drug_discovery_grpo.ipynb'),
           pathlib.Path('meta-hack.ipynb')]:
    if nb.exists():
        _copy(nb, 'notebook.ipynb')
        break

ckpt_dir = pathlib.Path('outputs/grpo')
if ckpt_dir.exists():
    INCLUDE_CHECKPOINT = bool(int(os.environ.get('INCLUDE_CHECKPOINT', '1')))
    if INCLUDE_CHECKPOINT:
        adapter_files = list(ckpt_dir.glob('adapter_*')) + list(ckpt_dir.glob('*.safetensors')) \
                      + list(ckpt_dir.glob('*.json')) + list(ckpt_dir.glob('tokenizer*'))
        for f in adapter_files:
            _copy(f, f'checkpoint/{f.name}')

zip_path = DEST_DIR / f'{bundle_name}.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=staging.parent, base_dir=bundle_name)
size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f'\nBundle ready: {zip_path}  ({size_mb:.1f} MB)')

if ON_KAGGLE:
    print('\nDownload on Kaggle:')
    print('  1. Open the right-hand "Output" panel of this notebook.')
    print(f'  2. Find {zip_path.name} and click the download icon.')
    try:
        from IPython.display import FileLink, display as _display
        _display(FileLink(str(zip_path)))
    except Exception:
        pass
else:
    try:
        from google.colab import files  # type: ignore
        files.download(str(zip_path))
    except Exception:
        try:
            from IPython.display import FileLink, display as _display
            _display(FileLink(str(zip_path)))
            print('Click the link above to download the bundle.')
        except Exception:
            print(f'Bundle saved at: {zip_path}')

In [24]:
!pip -q install google-api-python-client google-auth

import json
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

ZIP_PATH  = "/kaggle/working/fold.zip"
FOLDER_ID = "PASTE_YOUR_DRIVE_FOLDER_ID_HERE"

sa_info = json.loads(UserSecretsClient().get_secret("GDRIVE_SA_JSON"))
creds = service_account.Credentials.from_service_account_info(
    sa_info, scopes=["https://www.googleapis.com/auth/drive.file"]
)
drive = build("drive", "v3", credentials=creds, cache_discovery=False)

media = MediaFileUpload(ZIP_PATH, mimetype="application/zip", resumable=True)
meta  = {"name": Path(ZIP_PATH).name, "parents": [FOLDER_ID]}

req = drive.files().create(body=meta, media_body=media, fields="id, name, webViewLink")
resp = None
while resp is None:
    status, resp = req.next_chunk()
    if status:
        print(f"uploading… {int(status.progress()*100)}%")

print("Uploaded:", resp["name"], "->", resp["webViewLink"])

Created: /kaggle/working/fold.zip (147.8 MB)


/kaggle/working/fold.zip

## 8. Shut down the env server

In [ ]:
try:
    server_proc.terminate()
    server_proc.wait(timeout=5)
    print('env server stopped (exit', server_proc.returncode, ')')
except Exception as exc:
    print('shutdown error:', exc)
    server_proc.kill()

## Outputs

Everything below is plain JSON / JSONL / PNG and can be downloaded from the
Kaggle notebook output panel:

- `data/diseases.jsonl` — cached disease/target/known-drug rows + `train`/`test` split
- `data/diseases.manifest.json` — row counts, sha256, fetch timestamp
- `outputs/grpo/` — trained checkpoint (HF format, or PEFT adapter)
- `outputs/grpo/plots/` — **live** training plots (loss, reward +/- std,
  per-reward-component) written by `TrainingPlotsCallback` after every
  optimizer step
- `outputs/grpo/metrics.csv` — every numeric scalar logged by the trainer
  (loss, reward, reward_std, `rewards/<func>` per step) in CSV form
- `outputs/grpo/logs/kaggle-run/episodes.jsonl` — single consolidated trace
  with `episode_start` / `turn` / `episode_end` records (one JSONL line each;
  no per-episode file fan-out)
- `outputs/grpo/logs/kaggle-run/runs.csv` — one row per training episode
  including `nominated_compound_id` and `nominated_smiles`
- `outputs/plots/` — copy of the live training plots plus
  `episode_rewards.png` (per-episode total/terminal reward)
- `outputs/eval/report.json` — aggregate evaluation panel
- `outputs/eval/per_disease.jsonl` — per-test-disease metrics
- `outputs/infer/<slug>.json` — inference summary + reasoning trace